Academic Integrity and Learning Statement

By submitting my work, I confirm that:

1. The code, analysis, and documentation in this notebook are my own work and reflect my own understanding.
2. I am prepared to explain all code and analysis included in this submission.

If I used assistance (e.g., AI tools, tutors, or other resources), I have:

- Clearly documented where and how external tools or resources were used in my solution.
- Included a copy of the interaction (e.g., AI conversation or tutoring notes) in an appendix.

I acknowledge that:

- I may be asked to explain any part of my code or analysis during evaluation.
- Misrepresenting assisted work as my own constitutes academic dishonesty and undermines my learning.

In [ ]:
# Import core python libraries
import numpy as np
import os, platform, multiprocessing, subprocess
import pandas as pd

import torch

# Import custom python files
from src.utils import logging as prjlogger
from src.config import settings as prjconfigs
from src.visualisation import plots as prjplots
from src.data import loader as dloader
from src.features import engineering as prjfengg

In [ ]:
# Enable auto-reload extension
%load_ext autoreload
# Automatically reload all modules before executing code
%autoreload 2
%reload_ext autoreload

In [ ]:
# Check software specs
dict_sw_version = {
    'python': os.popen('python --version').read().strip(),
    'host platform': platform.platform(),
    'numpy': np.__version__,
    'pandas': pd.__version__
}

for key, value in dict_sw_version.items():
    print(f'{prjplots.beautify(key, 1)} version is: {prjplots.beautify(value)}')

In [ ]:
logger = prjlogger.get_logger()
logger.debug(f"Initialised logging for {prjconfigs.PROJECT_NAME} project.")

In [ ]:
# Check CPU cores
print(f'CPU cores available to use: {prjplots.beautify(str(multiprocessing.cpu_count()))}')

# Machine info
print(f"Processor: {prjplots.beautify(platform.processor())}")
print(f"Machine: {prjplots.beautify(platform.machine())}")

# GPU info
print("PyTorch MPS (Metal) Status:")
print(f"MPS available: {prjplots.beautify(torch.backends.mps.is_available())}")
print(f"MPS built: {prjplots.beautify(str(torch.backends.mps.is_built()))}")
print(f"CUDA available: {prjplots.beautify(str(torch.cuda.is_available()))}")

In [ ]:
df_raw_train = dloader.load_data(prjconfigs.TRAIN_FILE, True)
df_raw_test = dloader.load_data(prjconfigs.TEST_FILE, True)

In [ ]:
df_raw_train.sample(3)

In [ ]:
# Columns tagging
insignificant_cols = ['Order', 'PID']
target_col = 'SalePrice'
ignorable_cols = insignificant_cols + [target_col]

temporal_cols_name_pattern = ['Yr', 'Year']

ordinal_cols = ['LotShape', 'Utilities', 'LandSlope', 'OverallQual', 'OverallCond', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

In [ ]:
# Merge train and test data
df_raw_all, df_raw_target = dloader.merge_train_test_data(
    df_raw_train,
    df_raw_test,
    insignificant_cols,
    target_col
)

In [ ]:
# Split into train and test based on the is_train column
df_train = df_raw_all[df_raw_all['is_train'] == 1].iloc[:, :-1]
df_test = df_raw_all[df_raw_all['is_train'] == 0].iloc[:, :-1]

In [ ]:
# Classify features
feature_categories = prjfengg.classify_columns(
    df=df_train,
    n_cat_threshold=prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_ABS_VAL,
    threshold_type=prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_TYPE_ABS,
    cols_to_ignore=ignorable_cols,
    temporal_cols_name_pattern=temporal_cols_name_pattern,
    ordinal_cols=ordinal_cols
)